# RAG with LangChain + `langchain_openai` —

This notebook demonstrates a complete **Retrieval-Augmented Generation (RAG)** workflow using current LangChain-style APIs.

### What is covered
- PDF document loading
- Text chunking
- OpenAI embeddings using `langchain_openai`
- Chroma vector database
- Retriever `.invoke()` usage
- Final RAG answers using `ChatOpenAI`
- MultiQueryRetriever
- Contextual Compression Retriever
- Optional Wikipedia RAG without `WikipediaLoader`
- A second banking/Basel RAG example
- Constitution of India RAG example
- Simple RAG validation with an LLM-as-a-judge

> The code uses `.env` for API keys. No API key should be hard-coded in the notebook.

## 0. Environment setup

**Google Colab:** add `OPENAI_API_KEY` in Secrets and enable notebook access. `OPEN_API_KEY` is also accepted as an alias.

**Local Jupyter:** create `.env` beside the notebook:

```text
OPENAI_API_KEY=your_openai_api_key_here
OPENAI_MODEL=gpt-4.1-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

Model variables are optional. Run the installation cell before loading secrets. Then upload `data.zip` in the new setup cell; locally place it beside this notebook. Existing document-download cells below are unchanged and will still run.


In [1]:
%pip install -qU \
    langchain \
    langchain-classic \
    langchain-community \
    langchain-openai \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

> If this is the first time you installed or upgraded these packages, restart the Jupyter kernel once before continuing.

In [2]:
import os
from dotenv import load_dotenv

try:
    from google.colab import userdata
except ImportError:
    IN_COLAB = False
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPEN_API_KEY")
else:
    IN_COLAB = True
    api_key = None
    for secret_name in ["OPENAI_API_KEY", "OPEN_API_KEY"]:
        try:
            api_key = userdata.get(secret_name)
            if api_key:
                break
        except userdata.SecretNotFoundError:
            continue

if not api_key:
    raise ValueError("Add OPENAI_API_KEY to Colab Secrets or your local .env file.")
os.environ["OPENAI_API_KEY"] = api_key

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
print("Chat model:", CHAT_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))


Chat model: gpt-4.1-mini
Embedding model: text-embedding-3-small
OPENAI_API_KEY loaded: True


## Upload and extract data.zip

Run this cell once. The archive is extracted into the current working folder, preserving its internal folders.

In [3]:
import zipfile

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Select data.zip.
    if "data.zip" not in uploaded:
        raise FileNotFoundError("Please upload a file named data.zip.")

if not os.path.isfile("data.zip"):
    raise FileNotFoundError("Place data.zip beside this notebook or upload it in Colab.")

with zipfile.ZipFile("data.zip") as archive:
    destination = os.path.abspath(".")
    for member in archive.namelist():
        target = os.path.abspath(os.path.join(destination, member))
        if os.path.commonpath([destination, target]) != destination:
            raise ValueError("The ZIP contains an invalid extraction path.")
    archive.extractall(destination)
    print("data.zip extracted. Files:")
    for name in archive.namelist():
        print(name)


Saving data.zip to data.zip
data.zip extracted. Files:
data/EMPLOYEE_AGREEMENT.pdf
data/conversation_cases.csv
data/evaluation_questions.csv


In [4]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "langchain",
    "langchain-classic",
    "langchain-community",
    "langchain-openai",
    "langchain-chroma",
    "langchain-text-splitters",
    "chromadb",
]

for package in packages:
    try:
        print(f"{package:28s} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:28s} NOT INSTALLED")

langchain                    1.4.3
langchain-classic            1.0.8
langchain-community          0.4.2
langchain-openai             1.6.7
langchain-chroma             1.1.0
langchain-text-splitters     1.1.3
chromadb                     1.5.9


## 1. Common imports and helper functions

Important modern API changes used in this notebook:

- `retriever.invoke(question)` instead of `get_relevant_documents(...)`
- `from langchain_chroma import Chroma`
- `from langchain_text_splitters import RecursiveCharacterTextSplitter`
- `ChatOpenAI` and `OpenAIEmbeddings` come from `langchain_openai`
- `create_retrieval_chain` is used instead of the older `RetrievalQA`

In [5]:
import shutil
from pathlib import Path

import pandas as pd
import requests

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

/tmp/ipykernel_414/2797566886.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

print("LLM and embedding objects created.")

LLM and embedding objects created.


In [7]:
def download_file(url, output_path):
    """Download a file and fail clearly if the HTTP request is unsuccessful."""
    output_path = Path(output_path)

    headers = {
        "User-Agent": "LangChain-RAG-Training-Demo/1.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )
    response.raise_for_status()

    output_path.write_bytes(response.content)

    print(
        f"Downloaded: {output_path} "
        f"({len(response.content):,} bytes)"
    )

    return output_path

In [8]:
def show_documents(documents, max_chars=700):
    """Print retrieved documents in a trainer-friendly format."""
    for i, doc in enumerate(documents, start=1):
        print("=" * 90)
        print(f"DOCUMENT {i}")
        print("Metadata:", doc.metadata)
        print("-" * 90)
        print(doc.page_content[:max_chars])
        if len(doc.page_content) > max_chars:
            print("...")

In [9]:
def create_rag_chain(retriever, model):
    """
    Connect:
        Question -> Retriever -> Relevant chunks -> ChatOpenAI -> Final answer
    """

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are a RAG assistant.
Answer the user's question using ONLY the supplied context.

Rules:
1. Do not invent information.
2. If the answer is not available in the context, say:
   "The answer is not available in the retrieved context."
3. Keep the answer clear and concise.

Context:
{context}"""
            ),
            ("human", "{input}")
        ]
    )

    document_chain = create_stuff_documents_chain(
        model,
        prompt
    )

    return create_retrieval_chain(
        retriever,
        document_chain
    )

In [10]:
def ask_rag(rag_chain, question, show_context=True):
    """
    Execute the RAG chain and print both the generated answer
    and the retrieved source chunks.
    """
    response = rag_chain.invoke(
        {"input": question}
    )

    print("QUESTION")
    print(question)

    print("\nANSWER")
    print(response["answer"])

    if show_context:
        print("\nRETRIEVED CONTEXT")
        for i, doc in enumerate(response["context"], start=1):
            source = doc.metadata.get("source", "N/A")
            page = doc.metadata.get("page", "N/A")
            print(
                f"{i}. source={source}, page={page}"
            )

    return response

# Example 1 — Employee Agreement RAG

Pipeline:

**PDF → Pages → Chunks → OpenAI Embeddings → Chroma → Retriever → ChatOpenAI → Answer**

## Step 1 — Download and load the Employee Agreement PDF

In [11]:
EMPLOYEE_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/Agreements/EMPLOYEE_AGREEMENT.pdf"
)

employee_pdf = download_file(
    EMPLOYEE_URL,
    "EMPLOYEE_AGREEMENT.pdf"
)

employee_loader = PyPDFLoader(
    str(employee_pdf)
)

employee_pages = employee_loader.load()

print("Pages loaded:", len(employee_pages))

Downloaded: EMPLOYEE_AGREEMENT.pdf (206,079 bytes)
Pages loaded: 13


In [12]:
full_text = "\n".join(
    page.page_content
    for page in employee_pages
)

print("Pages      :", len(employee_pages))
print("Lines      :", len(full_text.splitlines()))
print("Words      :", len(full_text.split()))
print("Characters :", len(full_text))

Pages      : 13
Lines      : 382
Words      : 4540
Characters : 29553


## Step 2 — Split the document into chunks

In [13]:
employee_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

employee_chunks = employee_splitter.split_documents(
    employee_pages
)

print("Chunks created:", len(employee_chunks))
print("\nFirst chunk:")
print(employee_chunks[0].page_content[:700])

Chunks created: 73

First chunk:
EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT
EXHIBIT 10.1
EMPLOYEE AGREEMENT
THIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,
Inc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,
Suite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in
its entirety the employee agreement dated August 15, 2014 between Employee and the Company.


## Step 3 — Create a sample OpenAI embedding

In [14]:
sample_vector = embeddings.embed_query(
    "What is the sick leave policy?"
)

print("Embedding dimensions:", len(sample_vector))
print("First 10 values:", sample_vector[:10])

Embedding dimensions: 1536
First 10 values: [-0.0252838134765625, -0.007511138916015625, 0.031463623046875, 0.004978179931640625, -0.0283203125, 0.05072021484375, -0.0194244384765625, 0.0004017353057861328, -0.0022678375244140625, 0.00733184814453125]


## Step 4 — Store the chunks in Chroma

With the current Chroma integration, specifying `persist_directory` is enough for local persistence.  
A separate `persist()` call is not required.

In [15]:
EMP_DB_DIR = "emp_rules_db_final"

shutil.rmtree(
    EMP_DB_DIR,
    ignore_errors=True
)

emp_rules_db = Chroma.from_documents(
    documents=employee_chunks,
    embedding=embeddings,
    collection_name="employee_agreement",
    persist_directory=EMP_DB_DIR
)

print("Employee vector database created.")

Employee vector database created.


## Step 5 — Retrieve relevant chunks

In [16]:
employee_retriever = emp_rules_db.as_retriever(
    search_kwargs={"k": 4}
)

retrieved_docs = employee_retriever.invoke(
    "What is the policy for sick leaves?"
)

print("Retrieved documents:", len(retrieved_docs))
show_documents(retrieved_docs)

Retrieved documents: 4
DOCUMENT 1
Metadata: {'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'page': 1, 'source': 'EMPLOYEE_AGREEMENT.pdf', 'moddate': '2024-04-03T12:51:00+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'producer': 'Skia/PDF m123', 'page_label': '2', 'total_pages': 13}
------------------------------------------------------------------------------------------
ARTICLE 5
Vacations and Other Paid Absences
5.1. Vacation Days.  Employee shall be entitled to the same paid vacation days each calendar year
during the term of this Employee Agreement as authorized by the Company for its other employees.
5.2. Holidays.   Employee shall be entitled to the same paid holidays as authorized by the Company
for its other employees.
5.3. Sick Days and Personal Absence Days.  Employee shall be entitled to the same number of paid
DOCUMENT 2
Metadata: {'creationdate': '2024-04-0

## Step 6 — Connect the retriever to `ChatOpenAI`

This is the actual **generation** part of RAG.

The LLM does not receive the whole PDF. It receives the chunks returned by the retriever.

In [17]:
employee_rag = create_rag_chain(
    employee_retriever,
    llm
)

employee_response = ask_rag(
    employee_rag,
    "What is the policy for sick leaves?"
)

QUESTION
What is the policy for sick leaves?

ANSWER
The policy for sick leaves is that the Employee shall be entitled to the same number of paid sick days as authorized by the Company for its other employees.

RETRIEVED CONTEXT
1. source=EMPLOYEE_AGREEMENT.pdf, page=1
2. source=EMPLOYEE_AGREEMENT.pdf, page=1
3. source=EMPLOYEE_AGREEMENT.pdf, page=1
4. source=EMPLOYEE_AGREEMENT.pdf, page=2


In [18]:
ask_rag(
    employee_rag,
    "What is the base compensation?"
)

QUESTION
What is the base compensation?

ANSWER
The base compensation is $14,583 per month.

RETRIEVED CONTEXT
1. source=EMPLOYEE_AGREEMENT.pdf, page=1
2. source=EMPLOYEE_AGREEMENT.pdf, page=2
3. source=EMPLOYEE_AGREEMENT.pdf, page=1
4. source=EMPLOYEE_AGREEMENT.pdf, page=12


{'input': 'What is the base compensation?',
 'context': [Document(id='4d7509ae-c976-4dab-99c0-5b5b183c43be', metadata={'creationdate': '2024-04-03T12:51:00+00:00', 'page_label': '2', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'moddate': '2024-04-03T12:51:00+00:00', 'page': 1, 'total_pages': 13, 'producer': 'Skia/PDF m123', 'title': 'EMPLOYEE AGREEMENT'}, page_content='to Employee not less frequently than bi-monthly, or as is consistent with the Company’s practice for its other\nemployees.  \n4.2. Other Compensation.  Employee shall receive other compensation as more fully described on\nAppendix B, attached hereto and made a part hereof.\n4.3. Withholding. All amounts due from the Company to the Employee hereunder shall be paid to the\nEmployee net of all taxes and other amounts which the Company is required to withhold by law.'),
  Document(id='525634ca-83fb-4995-bb08-

In [19]:
ask_rag(
    employee_rag,
    "What does the agreement say about insurance?"
)

QUESTION
What does the agreement say about insurance?

ANSWER
The agreement states that the Company may provide employee benefits, including the Company's health insurance plan. However, no amounts paid to the Employee from an employee benefit plan shall count as compensation due to the Employee as base salary or additional compensation. Additionally, the Company is allowed to modify or terminate any of its employee benefit plans in a manner that does not discriminate between the Employee and other Company employees.

RETRIEVED CONTEXT
1. source=EMPLOYEE_AGREEMENT.pdf, page=2
2. source=EMPLOYEE_AGREEMENT.pdf, page=8
3. source=EMPLOYEE_AGREEMENT.pdf, page=5
4. source=EMPLOYEE_AGREEMENT.pdf, page=6


{'input': 'What does the agreement say about insurance?',
 'context': [Document(id='525634ca-83fb-4995-bb08-885760e18249', metadata={'producer': 'Skia/PDF m123', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'title': 'EMPLOYEE AGREEMENT', 'page_label': '3', 'moddate': '2024-04-03T12:51:00+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'page': 2, 'creationdate': '2024-04-03T12:51:00+00:00', 'total_pages': 13}, page_content='Company for its employees, including, without limitation, the Company’s health insurance plan. No amounts\npaid to Employee from an employee benefit plan shall count as compensation due Employee as base salary or\nadditional compensation.  Nothing in this Employee Agreement shall prohibit the Company from modifying\nor terminating any of its employee benefit plans in a manner that does not discriminate between Employee\nand other Company employees.\nARTICLE 7\nTermination of Employment'),
  

# Example 2 — Basel / Banking RAG

This example uses another PDF, but the same architecture:

**Banking PDF → chunks → embeddings → vector DB → retrieval → ChatOpenAI answer**

In [20]:
BASEL_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/Banking_System_Doc/BASEL.pdf"
)

basel_pdf = download_file(
    BASEL_URL,
    "BASEL.pdf"
)

basel_pages = PyPDFLoader(
    str(basel_pdf)
).load()

print("Pages loaded:", len(basel_pages))

Downloaded: BASEL.pdf (1,260,197 bytes)
Pages loaded: 77


In [21]:
basel_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

basel_chunks = basel_splitter.split_documents(
    basel_pages
)

print("Chunks created:", len(basel_chunks))

Chunks created: 408


In [22]:
BASEL_DB_DIR = "basel_norms_db_final"

shutil.rmtree(
    BASEL_DB_DIR,
    ignore_errors=True
)

basel_db = Chroma.from_documents(
    documents=basel_chunks,
    embedding=embeddings,
    collection_name="basel_rules",
    persist_directory=BASEL_DB_DIR
)

basel_retriever = basel_db.as_retriever(
    search_kwargs={"k": 5}
)

print("Basel vector database and retriever created.")

Basel vector database and retriever created.


In [23]:
basel_docs = basel_retriever.invoke(
    "What percentage is the minimum capital requirement?"
)

show_documents(
    basel_docs,
    max_chars=500
)

DOCUMENT 1
Metadata: {'creationdate': '2011-05-31T11:52:06+02:00', 'comments': '', 'total_pages': 77, 'doc #': '', 'moddate': '2019-11-14T17:33:29+01:00', 'description0': '', 'keywords': '', 'agenda item1': '', 'subject': 'Full text of "Basel III: A global regulatory framework for more resilient banks and banking systems - post BCBS meeting - revised version", June 2011', 'creator': 'Acrobat PDFMaker 9.1 for Word', 'page_label': '64', 'producer': 'Acrobat Distiller 9.4.0 (Windows)', 'source': 'BASEL.pdf', 'title': 'Basel III: A global regulatory framework for more resilient banks and banking systems - post BCBS meeting - revised version June 2011', 'author': 'Basel Committee on Banking Supervision', 'company': '', 'sort order': '0.110000000000000', 'page': 63, 'meeting': '2010 - 138th Meeting of the BCBS November-December 2010', 'contenttype': 'Document'}
------------------------------------------------------------------------------------------
and no Additional Tier 1 or Tier 2 capita

In [24]:
basel_rag = create_rag_chain(
    basel_retriever,
    llm
)

ask_rag(
    basel_rag,
    "What percentage is the minimum capital requirement?"
)

QUESTION
What percentage is the minimum capital requirement?

ANSWER
The minimum capital requirements in relation to risk-weighted assets (RWAs) are:
- 3.5% Common Equity Tier 1/RWAs
- 4.5% Tier 1 capital/RWAs
- 8.0% total capital/RWAs

RETRIEVED CONTEXT
1. source=BASEL.pdf, page=63
2. source=BASEL.pdf, page=66
3. source=BASEL.pdf, page=76
4. source=BASEL.pdf, page=62
5. source=BASEL.pdf, page=34


{'input': 'What percentage is the minimum capital requirement?',
 'context': [Document(id='91070e28-c46c-4f13-9eac-b6b554d5409e', metadata={'author': 'Basel Committee on Banking Supervision', 'title': 'Basel III: A global regulatory framework for more resilient banks and banking systems - post BCBS meeting - revised version June 2011', 'agenda item1': '', 'producer': 'Acrobat Distiller 9.4.0 (Windows)', 'creationdate': '2011-05-31T11:52:06+02:00', 'subject': 'Full text of "Basel III: A global regulatory framework for more resilient banks and banking systems - post BCBS meeting - revised version", June 2011', 'meeting': '2010 - 138th Meeting of the BCBS November-December 2010', 'comments': '', 'creator': 'Acrobat PDFMaker 9.1 for Word', 'moddate': '2019-11-14T17:33:29+01:00', 'total_pages': 77, 'sort order': '0.110000000000000', 'contenttype': 'Document', 'source': 'BASEL.pdf', 'company': '', 'page_label': '64', 'page': 63, 'keywords': '', 'description0': '', 'doc #': ''}, page_content=

In [25]:
ask_rag(
    basel_rag,
    "What are PD and LGD?"
)

QUESTION
What are PD and LGD?

ANSWER
PD stands for Probability of Default, and LGD stands for Loss Given Default.

RETRIEVED CONTEXT
1. source=BASEL.pdf, page=46
2. source=BASEL.pdf, page=46
3. source=BASEL.pdf, page=46
4. source=BASEL.pdf, page=6
5. source=BASEL.pdf, page=13


{'input': 'What are PD and LGD?',
 'context': [Document(id='5f2b9cf2-4c27-40c1-bd51-4cec612cb69a', metadata={'page_label': '47', 'moddate': '2019-11-14T17:33:29+01:00', 'sort order': '0.110000000000000', 'creationdate': '2011-05-31T11:52:06+02:00', 'author': 'Basel Committee on Banking Supervision', 'producer': 'Acrobat Distiller 9.4.0 (Windows)', 'meeting': '2010 - 138th Meeting of the BCBS November-December 2010', 'doc #': '', 'agenda item1': '', 'contenttype': 'Document', 'subject': 'Full text of "Basel III: A global regulatory framework for more resilient banks and banking systems - post BCBS meeting - revised version", June 2011', 'creator': 'Acrobat PDFMaker 9.1 for Word', 'company': '', 'page': 46, 'description0': '', 'keywords': '', 'source': 'BASEL.pdf', 'title': 'Basel III: A global regulatory framework for more resilient banks and banking systems - post BCBS meeting - revised version June 2011', 'total_pages': 77, 'comments': ''}, page_content='contain an LGD assumption, the

# Example 3 — MultiQueryRetriever + RAG

A standard retriever searches with one user query.

`MultiQueryRetriever` asks the LLM to create multiple alternative versions of the question, runs retrieval for them, and combines the unique results.

For current LangChain versions it is provided through **`langchain_classic`**.



MultiQueryRetriever is a RAG technique that searches using multiple versions of your question. This helps find relevant information even when your wording differs from the documents.
Simple banking example
You ask:
“What happens if I miss my loan EMI?”

The LLM creates related search questions:
- “What are the penalties for a late EMI payment?”
- “How does a missed EMI affect the credit score?”
- “What happens when a loan instalment is overdue?”
The retriever searches your documents for each question, combines the retrieved chunks, and removes duplicates. The LLM then uses those chunks to answer you

In [26]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=emp_rules_db.as_retriever(
        search_kwargs={"k": 3}
    ),
    llm=llm,
    include_original=True
)

print("MultiQueryRetriever created.")

MultiQueryRetriever created.


In [27]:
multi_question = (
    "What leave benefits are available "
    "when an employee is sick?"
)

multi_docs = multiquery_retriever.invoke(
    multi_question
)

print("Unique documents returned:", len(multi_docs))
show_documents(
    multi_docs,
    max_chars=500
)

Unique documents returned: 3
DOCUMENT 1
Metadata: {'page_label': '2', 'title': 'EMPLOYEE AGREEMENT', 'total_pages': 13, 'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'page': 1, 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf'}
------------------------------------------------------------------------------------------
ARTICLE 5
Vacations and Other Paid Absences
5.1. Vacation Days.  Employee shall be entitled to the same paid vacation days each calendar year
during the term of this Employee Agreement as authorized by the Company for its other employees.
5.2. Holidays.   Employee shall be entitled to the same paid holidays as authorized by the Company
for its other employees.
5.3. Sick Days and Personal Absence Days.  Employee shall be entitled to the same number of paid
DOCUMENT 2
Metadata: {'creationdate': '202

## Connect MultiQueryRetriever to ChatOpenAI

In [28]:
multiquery_rag = create_rag_chain(
    multiquery_retriever,
    llm
)

ask_rag(
    multiquery_rag,
    multi_question
)

QUESTION
What leave benefits are available when an employee is sick?

ANSWER
When an employee is sick, they are entitled to the same number of paid sick days as authorized by the Company for its other employees.

RETRIEVED CONTEXT
1. source=EMPLOYEE_AGREEMENT.pdf, page=1
2. source=EMPLOYEE_AGREEMENT.pdf, page=1
3. source=EMPLOYEE_AGREEMENT.pdf, page=1


{'input': 'What leave benefits are available when an employee is sick?',
 'context': [Document(id='c6eb95aa-ddaf-487e-8b95-bab8fa8e2ed3', metadata={'producer': 'Skia/PDF m123', 'page': 1, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'total_pages': 13, 'source': 'EMPLOYEE_AGREEMENT.pdf', 'moddate': '2024-04-03T12:51:00+00:00', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'page_label': '2'}, page_content='ARTICLE 5\nVacations and Other Paid Absences\n5.1. Vacation Days.  Employee shall be entitled to the same paid vacation days each calendar year\nduring the term of this Employee Agreement as authorized by the Company for its other employees.\n5.2. Holidays.   Employee shall be entitled to the same paid holidays as authorized by the Company\nfor its other employees.\n5.3. Sick Days and Personal Absence Days.  Employee shall be entitled to the same number of paid'),
  Document(i

# Example 4 — Contextual Compression Retriever + RAG

Contextual compression performs two steps:

1. Retrieve potentially relevant chunks.
2. Use the LLM to extract only the portions useful for the current question.

This can reduce irrelevant text before the final answer-generation step.

In [29]:
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever
)
from langchain_classic.retrievers.document_compressors.chain_extract import (
    LLMChainExtractor
)

base_retriever = emp_rules_db.as_retriever(
    search_kwargs={"k": 5}
)

compressor = LLMChainExtractor.from_llm(
    llm
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("Contextual compression retriever created.")

Contextual compression retriever created.


In [30]:
compression_question = (
    "What is the policy for sick leave?"
)

compressed_docs = compression_retriever.invoke(
    compression_question
)

print("Compressed documents:", len(compressed_docs))
show_documents(
    compressed_docs,
    max_chars=500
)

Compressed documents: 3
DOCUMENT 1
Metadata: {'title': 'EMPLOYEE AGREEMENT', 'page_label': '2', 'page': 1, 'total_pages': 13, 'creationdate': '2024-04-03T12:51:00+00:00', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'}
------------------------------------------------------------------------------------------
sick days and personal absence days as authorized by the Company for its other employees.
DOCUMENT 2
Metadata: {'moddate': '2024-04-03T12:51:00+00:00', 'page': 1, 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'title': 'EMPLOYEE AGREEMENT', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'producer': 'Skia/PDF m123', 'page_label': '2'}
---------------------------------------

## Connect the compressed retriever to ChatOpenAI

In [31]:
compression_rag = create_rag_chain(
    compression_retriever,
    llm
)

ask_rag(
    compression_rag,
    compression_question
)

QUESTION
What is the policy for sick leave?

ANSWER
The policy for sick leave states that the employee shall be entitled to the same number of paid sick days and personal absence days as authorized by the Company for its other employees.

RETRIEVED CONTEXT
1. source=EMPLOYEE_AGREEMENT.pdf, page=1
2. source=EMPLOYEE_AGREEMENT.pdf, page=1
3. source=EMPLOYEE_AGREEMENT.pdf, page=1


{'input': 'What is the policy for sick leave?',
 'context': [Document(metadata={'moddate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'total_pages': 13, 'page': 1, 'producer': 'Skia/PDF m123', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'creationdate': '2024-04-03T12:51:00+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'page_label': '2'}, page_content='sick days and personal absence days as authorized by the Company for its other employees.'),
  Document(metadata={'moddate': '2024-04-03T12:51:00+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'total_pages': 13, 'page': 1, 'creationdate': '2024-04-03T12:51:00+00:00', 'page_label': '2', 'producer': 'Skia/PDF m123', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'title': 'EMPLOYEE AGREEMENT'}, page_content='5.3. Sick Days and Personal Absence Days.  Employee shall be ent

# Optional Example — Wikipedia RAG without `WikipediaLoader`

The older `wikipedia` package can fail with `JSONDecodeError` when the API response is blocked or is not JSON.

This optional example calls the MediaWiki API directly using `requests`, supplies a User-Agent, converts the result into LangChain `Document` objects, and then uses the same RAG pipeline.

If Wikipedia is blocked by a corporate proxy/firewall, the cell prints a message and the rest of the notebook can still run.

In [32]:
def load_wikipedia_documents(query, max_docs=2):
    api_url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": (
            "LangChain-RAG-Training-Demo/1.0 "
            "(educational notebook)"
        )
    }

    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "utf8": 1,
        "srlimit": max_docs
    }

    search_response = requests.get(
        api_url,
        params=search_params,
        headers=headers,
        timeout=30
    )
    search_response.raise_for_status()

    try:
        search_data = search_response.json()
    except ValueError as exc:
        preview = search_response.text[:250]
        raise RuntimeError(
            "Wikipedia did not return JSON. "
            f"Response preview: {preview!r}"
        ) from exc

    search_results = search_data.get(
        "query",
        {}
    ).get(
        "search",
        []
    )

    documents = []

    for result in search_results:
        title = result["title"]

        extract_params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "redirects": 1,
            "titles": title,
            "format": "json",
            "utf8": 1
        }

        extract_response = requests.get(
            api_url,
            params=extract_params,
            headers=headers,
            timeout=30
        )
        extract_response.raise_for_status()

        extract_data = extract_response.json()

        pages = extract_data.get(
            "query",
            {}
        ).get(
            "pages",
            {}
        )

        for page in pages.values():
            text = page.get("extract", "").strip()

            if text:
                documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "title": page.get(
                                "title",
                                title
                            ),
                            "source": (
                                "https://en.wikipedia.org/wiki/"
                                + title.replace(" ", "_")
                            )
                        }
                    )
                )

    return documents

In [33]:
try:
    wiki_documents = load_wikipedia_documents(
        "MS Dhoni",
        max_docs=2
    )

    print(
        "Wikipedia documents loaded:",
        len(wiki_documents)
    )

except Exception as exc:
    wiki_documents = []

    print(
        "Wikipedia API is unavailable in this "
        "environment, so this optional example "
        "will be skipped."
    )
    print("Reason:", exc)

Wikipedia documents loaded: 2


In [34]:
if wiki_documents:
    wiki_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )

    wiki_chunks = wiki_splitter.split_documents(
        wiki_documents
    )

    WIKI_DB_DIR = "wiki_db_final"

    shutil.rmtree(
        WIKI_DB_DIR,
        ignore_errors=True
    )

    wiki_db = Chroma.from_documents(
        documents=wiki_chunks,
        embedding=embeddings,
        collection_name="wikipedia_demo",
        persist_directory=WIKI_DB_DIR
    )

    wiki_retriever = wiki_db.as_retriever(
        search_kwargs={"k": 4}
    )

    wiki_rag = create_rag_chain(
        wiki_retriever,
        llm
    )

    ask_rag(
        wiki_rag,
        "What is MS Dhoni's date of birth?"
    )
else:
    print(
        "Skipping Wikipedia vector DB and RAG "
        "because no Wikipedia documents were loaded."
    )

QUESTION
What is MS Dhoni's date of birth?

ANSWER
MS Dhoni's date of birth is 7 July 1981.

RETRIEVED CONTEXT
1. source=https://en.wikipedia.org/wiki/MS_Dhoni, page=N/A
2. source=https://en.wikipedia.org/wiki/MS_Dhoni, page=N/A
3. source=https://en.wikipedia.org/wiki/MS_Dhoni, page=N/A
4. source=https://en.wikipedia.org/wiki/MS_Dhoni, page=N/A


# Example 5 — Constitution of India RAG

This section keeps the Constitution-of-India use case, but uses the same modern RAG architecture and `ChatOpenAI`.

In [35]:
COI_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/COI/COI.pdf"
)

coi_pdf = download_file(
    COI_URL,
    "COI.pdf"
)

coi_pages = PyPDFLoader(
    str(coi_pdf)
).load()

print("Pages loaded:", len(coi_pages))

Downloaded: COI.pdf (3,922,267 bytes)
Pages loaded: 402


In [36]:
coi_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=300
)

coi_chunks = coi_splitter.split_documents(
    coi_pages
)

print("Chunks created:", len(coi_chunks))

Chunks created: 714


In [37]:
COI_DB_DIR = "coi_db_final"

shutil.rmtree(
    COI_DB_DIR,
    ignore_errors=True
)

coi_db = Chroma.from_documents(
    documents=coi_chunks,
    embedding=embeddings,
    collection_name="constitution_of_india",
    persist_directory=COI_DB_DIR
)

coi_retriever = coi_db.as_retriever(
    search_kwargs={"k": 5}
)

coi_rag = create_rag_chain(
    coi_retriever,
    llm
)

print("Constitution RAG chain created.")

Constitution RAG chain created.


In [38]:
coi_response = ask_rag(
    coi_rag,
    (
        "According to the Constitution of India, "
        "what are the fundamental rights of citizens?"
    )
)

QUESTION
According to the Constitution of India, what are the fundamental rights of citizens?

ANSWER
According to the Constitution of India, the fundamental rights of citizens include:

1. Right to Equality:
   - Equality before the law (Article 14)
   - Prohibition of discrimination on grounds of religion, race, caste, sex, or place of birth (Article 15)
   - Equality of opportunity in public employment (Article 16)
   - Abolition of Untouchability (Article 17)
   - Abolition of titles (Article 18)

2. Right to Freedom:
   - Freedom of speech and expression
   - Right to assemble peaceably and without arms
   - Right to form associations or unions or co-operative societies
   - Right to move freely throughout the territory of India (Article 19)

3. Protection in respect of conviction for offences (Article 20)

4. Protection of life and personal liberty (Article 21)

5. Right to education (Article 21A)

6. Protection against arrest and detention in certain cases (Article 22)

7. Right

## Out-of-context test

A well-instructed RAG chain should avoid answering from general model knowledge when the retrieved context does not support the answer.

In [39]:
ask_rag(
    coi_rag,
    "What is the current weather in Hyderabad?"
)

QUESTION
What is the current weather in Hyderabad?

ANSWER
The answer is not available in the retrieved context.

RETRIEVED CONTEXT
1. source=COI.pdf, page=267
2. source=COI.pdf, page=270
3. source=COI.pdf, page=276
4. source=COI.pdf, page=275
5. source=COI.pdf, page=303


{'input': 'What is the current weather in Hyderabad?',
 'context': [Document(id='fd41e261-5197-4431-b247-b5083c78a680', metadata={'page': 267, 'page_label': '268', 'creationdate': '2022-01-14T06:43:16+00:00', 'moddate': '2022-01-14T06:43:17+00:00', 'creator': 'Microsoft® Word 2016', 'total_pages': 402, 'author': 'DELL', 'source': 'COI.pdf', 'producer': 'www.ilovepdf.com'}, page_content='Pradesh or the State of Telangana].—5[(1) The President may by order made \nwith respect  to the State of Andhra Pradesh or the State of  Telangana, provide, \nhaving regard to the requirement of each State, for equitable opportunities and \nfacilities for the people belonging to different parts of such State, in the matter \nof public employment and in the matter of education, and different provisions \nmay be made for various parts of the States.] \n(2) An order made under clause (1) may, in particular,— \n                                                             \n1. Subs. by the North -Eastern Ar

# Example 6 — Simple RAG validation using `ChatOpenAI`

Instead of the older `QAEvalChain`, this section uses a current runnable pipeline:

**Reference answer + RAG answer → Evaluation prompt → ChatOpenAI → PASS/FAIL + reason**

To keep API usage small, only the first few CSV rows are evaluated by default.

In [40]:
COI_QA_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/COI/COI_Q_A.csv"
)

qa_csv = download_file(
    COI_QA_URL,
    "COI_Q_A.csv"
)

test_data = pd.read_csv(
    qa_csv
)

display(test_data.head())

Downloaded: COI_Q_A.csv (5,143 bytes)


,Question,Answer
0,According to constitution of India what are th...,The fundamental rights include the Right to Eq...
1,What is the significance of the Preamble in th...,The Preamble serves as an introduction to the ...
2,How does the Constitution of India ensure the ...,The Constitution ensures the independence of t...
3,What are the Directive Principles of State Pol...,The Directive Principles of State Policy are n...
4,What is the concept of 'Secularism' in the con...,"In the Indian Constitution, 'Secularism' means..."


In [41]:
judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are evaluating a RAG answer.

Compare the generated answer with the reference answer.

Return exactly two lines:
Verdict: PASS or FAIL
Reason: one short reason

A PASS means the generated answer is substantively
consistent with the reference answer. Minor wording
differences are acceptable."""
        ),
        (
            "human",
            """Question:
{question}

Reference answer:
{reference_answer}

Generated RAG answer:
{generated_answer}"""
        )
    ]
)

judge_chain = (
    judge_prompt
    | llm
    | StrOutputParser()
)

print("RAG evaluation chain created.")

RAG evaluation chain created.


In [42]:
MAX_EVAL_ROWS = min(
    5,
    len(test_data)
)

evaluation_rows = []

for _, row in test_data.head(MAX_EVAL_ROWS).iterrows():
    question = str(row["Question"])
    reference_answer = str(row["Answer"])

    rag_response = coi_rag.invoke(
        {"input": question}
    )

    generated_answer = rag_response[
        "answer"
    ]

    evaluation = judge_chain.invoke(
        {
            "question": question,
            "reference_answer": reference_answer,
            "generated_answer": generated_answer
        }
    )

    evaluation_rows.append(
        {
            "Question": question,
            "Reference Answer": reference_answer,
            "RAG Answer": generated_answer,
            "Evaluation": evaluation
        }
    )

evaluation_df = pd.DataFrame(
    evaluation_rows
)

display(evaluation_df)

,Question,Reference Answer,RAG Answer,Evaluation
0,According to constitution of India what are th...,The fundamental rights include the Right to Eq...,"According to the Constitution of India, the fu...",Verdict: PASS\nReason: The generated answer co...
1,What is the significance of the Preamble in th...,The Preamble serves as an introduction to the ...,The Preamble of the Constitution of India sole...,Verdict: PASS \nReason: The generated answer ...
2,How does the Constitution of India ensure the ...,The Constitution ensures the independence of t...,The Constitution of India ensures the independ...,Verdict: PASS \nReason: The generated answer ...
3,What are the Directive Principles of State Pol...,The Directive Principles of State Policy are n...,"The Directive Principles of State Policy, outl...",Verdict: PASS \nReason: The generated answer ...
4,What is the concept of 'Secularism' in the con...,"In the Indian Constitution, 'Secularism' means...",The concept of 'Secularism' in the context of ...,Verdict: PASS \nReason: The generated answer ...


In [43]:
pass_count = (
    evaluation_df["Evaluation"]
    .str.contains(
        "Verdict: PASS",
        case=False,
        na=False
    )
    .sum()
)

total = len(evaluation_df)

print(f"PASS results: {pass_count}/{total}")

if total:
    print(
        "Approximate pass rate:",
        f"{(pass_count / total) * 100:.1f}%"
    )

PASS results: 5/5
Approximate pass rate: 100.0%


# Final architecture recap

For every example in this notebook, the main RAG flow is:

```text
User Question
      |
      v
Retriever
      |
      v
Relevant Chunks from Chroma
      |
      v
Prompt + Retrieved Context
      |
      v
ChatOpenAI
      |
      v
Grounded Final Answer
```

### Key objects used

```python
OpenAIEmbeddings(...)
Chroma.from_documents(...)
vector_db.as_retriever(...)
retriever.invoke(...)
ChatOpenAI(...)
create_stuff_documents_chain(...)
create_retrieval_chain(...)
rag_chain.invoke({"input": question})
```

### MultiQuery version

```text
Question
   |
   v
ChatOpenAI generates alternative queries
   |
   v
MultiQueryRetriever
   |
   v
Combined unique chunks
   |
   v
ChatOpenAI generates final RAG answer
```

### Contextual compression version

```text
Question
   |
   v
Base Retriever
   |
   v
Candidate chunks
   |
   v
LLMChainExtractor
   |
   v
Compressed relevant context
   |
   v
ChatOpenAI generates final RAG answer
```